# ⚡ Electromagnetism — Interactive Visualizations (Ch 1–7)
**PSE Lab** · Background notes for the Griffiths QM course (after D. Tong, *Lectures on Electromagnetism*) · Run each cell to generate interactive plots (sliders need a live kernel, e.g. Colab)

| Chapter | Visualization |
|---|---|
| Ch 1–2 | Coulomb field lines & equipotentials |
| Ch 3 | Magnetic field of a current loop (3D) |
| Ch 4 | EM wave propagation (animated) |
| Ch 5 | Lorentz boost: E & B field transformation |
| Ch 6 | Dipole radiation pattern & Larmor power vs ω |
| Ch 7 | Lorentz oscillator ε(ω): dispersion & absorption |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
import warnings; warnings.filterwarnings('ignore')

# Dark style consistent with slides
plt.rcParams.update({
    'figure.facecolor':'#1a1a2e','axes.facecolor':'#16213e',
    'axes.edgecolor':'#4a5a7e','axes.labelcolor':'#f0f0f0',
    'xtick.color':'#aaa','ytick.color':'#aaa',
    'text.color':'#f0f0f0','grid.color':'#2a3a5e',
    'grid.linestyle':'--','grid.alpha':0.5
})
BLUE,GREEN,YELLOW,RED,DIM = '#7ec8e3','#90ee90','#f9c74f','#e07070','#aaaaaa'
print('Setup complete ✓')

## Ch 2: Coulomb Field Lines & Equipotentials
Adjust charge positions and magnitudes to see how field lines and equipotentials change.

In [ ]:
def plot_field(q1=1.0, q2=-1.0, x1=-1.0, x2=1.0):
    fig, ax = plt.subplots(figsize=(8,7), facecolor='#1a1a2e')
    ax.set_facecolor('#16213e')
    ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_aspect('equal')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(f'Coulomb field  q₁={q1:+.1f}  q₂={q2:+.1f}', color=BLUE, fontsize=13)

    X,Y = np.meshgrid(np.linspace(-3,3,400), np.linspace(-3,3,400))
    r1 = np.sqrt((X-x1)**2+Y**2)+1e-9
    r2 = np.sqrt((X-x2)**2+Y**2)+1e-9
    phi = q1/r1 + q2/r2

    # Equipotentials
    levels = np.linspace(-4,4,25)
    ax.contour(X,Y,np.clip(phi,-4,4), levels=levels,
               colors=[BLUE], alpha=0.4, linewidths=0.8)

    # Field lines via streamplot
    Ex = q1*(X-x1)/r1**3 + q2*(X-x2)/r2**3
    Ey = q1*Y/r1**3       + q2*Y/r2**3
    speed = np.sqrt(Ex**2+Ey**2)+1e-9
    ax.streamplot(X[0],Y[:,0],Ex,Ey, color=GREEN,
                  linewidth=0.8, density=1.4, arrowsize=0.8)

    # Charges
    for xc,q,lbl in [(x1,q1,'q₁'),(x2,q2,'q₂')]:
        c = YELLOW if q>0 else RED
        ax.plot(xc,0,'o',color=c,ms=14,zorder=5)
        ax.text(xc,0.25,f'{lbl}={q:+.1f}',color=c,ha='center',fontsize=10)

    ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()

interact(plot_field,
    q1=FloatSlider(value=1.0, min=-3, max=3, step=0.5, description='q₁'),
    q2=FloatSlider(value=-1.0, min=-3, max=3, step=0.5, description='q₂'),
    x1=FloatSlider(value=-1.0, min=-2, max=0, step=0.25, description='x₁'),
    x2=FloatSlider(value=1.0, min=0, max=2, step=0.25, description='x₂'));

## Ch 3: Magnetic Field of a Current Loop
The on-axis field is $B_z = \mu_0 I R^2 / 2(R^2+z^2)^{3/2}$. Adjust loop radius and current.

In [ ]:
def plot_bfield(R=1.0, I=1.0):
    fig, axes = plt.subplots(1,2,figsize=(12,5),facecolor='#1a1a2e')

    # Left: 2D cross-section streamplot
    ax = axes[0]; ax.set_facecolor('#16213e')
    ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_aspect('equal')
    ax.set_title('B field (cross-section, μ₀=1)', color=BLUE)
    ax.set_xlabel('r (radial)'); ax.set_ylabel('z')

    x,z = np.meshgrid(np.linspace(-3,3,200), np.linspace(-3,3,200))
    # Approximate dipole field for far field
    r2 = x**2+z**2+1e-9; r5 = r2**2.5; r3 = r2**1.5
    m = I*np.pi*R**2  # magnetic moment
    Bx = 3*x*z*m/r5
    Bz = (3*z**2-r2)*m/r5
    # near-field: mask inside coil region
    mask = (np.abs(x)<R*0.3) & (np.abs(z)<0.15)
    Bx[mask]=0; Bz[mask]=0
    ax.streamplot(x[0],z[:,0],Bx,Bz,color=GREEN,density=1.3,linewidth=0.9,arrowsize=0.8)
    # Draw loop
    ax.plot([-R,R],[0,0],color=YELLOW,lw=3,solid_capstyle='round')
    ax.text(0,0.2,f'I={I:.1f}',color=YELLOW,ha='center',fontsize=10)
    ax.plot(0,0,'o',color=YELLOW,ms=8)
    ax.grid(True,alpha=0.3)

    # Right: on-axis B vs z
    ax2 = axes[1]; ax2.set_facecolor('#16213e')
    z_ax = np.linspace(-4,4,300)
    Bz_ax = I*R**2/(2*(R**2+z_ax**2)**1.5)
    Bdip  = I*R**2/(2*np.abs(z_ax)**3+1e-9)  # dipole: mu0*2m/(4pi z^3), m = I*pi*R^2, mu0 = 1
    ax2.plot(z_ax, Bz_ax, color=GREEN,  lw=2, label='Exact on-axis')
    ax2.plot(z_ax, Bdip,  color=YELLOW, lw=1.2, ls='--', label='Dipole approx')
    ax2.axvline(R,  color=DIM,lw=1,ls=':'); ax2.axvline(-R,color=DIM,lw=1,ls=':')
    ax2.set_xlabel('z / R'); ax2.set_ylabel('Bz (μ₀=1)')
    ax2.set_title('On-axis B field', color=BLUE)
    ax2.set_ylim(-0.1, I/(2*R**2)*1.2)
    ax2.legend(facecolor='#1a1a2e',labelcolor='white')
    ax2.grid(True,alpha=0.3)
    ax2.text(0, I/(2*R**2)*1.05, f'B_max={I/(2*R**2):.3f} at z=0',
             color=GREEN, ha='center', fontsize=9)
    plt.suptitle(f'Magnetic field of current loop   R={R:.1f}  I={I:.1f}',
                 color=BLUE, fontsize=13)
    plt.tight_layout(); plt.show()

interact(plot_bfield,
    R=FloatSlider(value=1.0, min=0.3, max=2.5, step=0.1, description='Radius R'),
    I=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Current I'));

## Ch 4: Electromagnetic Wave
Plane wave $\mathbf{E}=E_0\hat{x}\cos(kz-\omega t)$, $\mathbf{B}=B_0\hat{y}\cos(kz-\omega t)$ with $|B_0|=|E_0|/c$.

In [ ]:
def plot_emwave(t_frac=0.0, polarization='Linear (x)'):
    fig = plt.figure(figsize=(11,5),facecolor='#1a1a2e')
    ax = fig.add_subplot(111,projection='3d')
    ax.set_facecolor('#1a1a2e')
    for p in [ax.xaxis.pane,ax.yaxis.pane,ax.zaxis.pane]:
        p.fill=False; p.set_edgecolor('#2a3a5e')
    ax.tick_params(colors=DIM,labelsize=7)

    z = np.linspace(0,4*np.pi,300)
    t = t_frac*2*np.pi

    if polarization=='Linear (x)':
        Ex,Ey = np.sin(z-t), np.zeros_like(z)
    elif polarization=='Linear (y)':
        Ex,Ey = np.zeros_like(z), np.sin(z-t)
    elif polarization=='Circular (R)':
        Ex,Ey = np.cos(z-t), np.sin(z-t)
    else:
        Ex,Ey = np.cos(z-t), 0.5*np.sin(z-t)

    # E field (red)
    ax.plot(Ex,Ey,z,color=RED,lw=2,label='E field')
    for i in range(0,len(z),25):
        ax.quiver(0,0,z[i],Ex[i],Ey[i],0,color=RED,alpha=0.7,
                  lw=0.8,arrow_length_ratio=0.15)

    # B field (perpendicular, blue) — only for linear x
    if polarization=='Linear (x)':
        ax.plot(np.zeros_like(z),np.sin(z-t),z,color=BLUE,lw=2,label='B field')
        for i in range(0,len(z),25):
            ax.quiver(0,0,z[i],0,np.sin(z[i]-t),0,color=BLUE,alpha=0.7,
                      lw=0.8,arrow_length_ratio=0.15)

    # Propagation axis
    ax.plot([0,0],[0,0],[0,4*np.pi+1],color=DIM,lw=1.5,alpha=0.7)
    ax.quiver(0,0,4*np.pi,0,0,0.8,color=DIM,arrow_length_ratio=0.4,lw=2)
    ax.text(0.1,0.1,4*np.pi+1.5,'k̂ (propagation)',color=DIM,fontsize=10)

    ax.set_xlabel('Ex',color=RED,fontsize=9)
    ax.set_ylabel('Ey / By',color=BLUE,fontsize=9)
    ax.set_zlabel('z',color=DIM,fontsize=9)
    ax.set_xlim(-1.3,1.3);ax.set_ylim(-1.3,1.3);ax.set_zlim(0,14)
    ax.set_xticks([-1,0,1]);ax.set_yticks([-1,0,1]);ax.set_zticks([])
    ax.view_init(elev=20,azim=-50)
    ax.set_title(f'EM Plane Wave  |  {polarization}  |  t={t_frac:.2f}·T',
                 color=BLUE,fontsize=11,pad=8)
    plt.tight_layout(); plt.show()

interact(plot_emwave,
    t_frac=FloatSlider(value=0.0, min=0.0, max=1.0, step=0.02, description='Time t/T'),
    polarization=Dropdown(options=['Linear (x)','Linear (y)',
                                    'Circular (R)','Elliptic'],
                           description='Polarization'));

## Ch 5: Lorentz Boost of E & B Fields
A pure electric field in frame S becomes both E and B in frame S' moving at speed v.
Demonstrates that **magnetism is a relativistic effect of electricity**.

In [ ]:
def plot_boost(beta=0.5, config='Point charge (E only)'):
    gamma = 1/np.sqrt(1-beta**2)
    fig, axes = plt.subplots(1,2,figsize=(12,5),facecolor='#1a1a2e')

    x,y = np.meshgrid(np.linspace(-3,3,300),np.linspace(-3,3,300))
    r = np.sqrt(x**2+y**2)+1e-9

    if config=='Point charge (E only)':
        # Lab frame: Coulomb
        Ex_lab = x/r**3; Ey_lab = y/r**3
        Bz_lab = np.zeros_like(x)
        # Boosted frame: E enhanced transverse, B appears
        Ex_b = Ex_lab
        Ey_b = gamma*Ey_lab
        Bz_b = -gamma*beta*Ey_lab  # B ~ v×E
    else:  # Line charge along z (E only, falls off as 1/r)
        Ex_lab = x/r**2; Ey_lab = y/r**2
        Bz_lab = np.zeros_like(x)
        # Boost along x: E'_x = E_x, E'_y = gamma E_y, B'_z = -gamma beta E_y (c = 1)
        Ex_b = Ex_lab
        Ey_b = gamma*Ey_lab
        Bz_b = -gamma*beta*Ey_lab

    for ax, (Ex,Ey), title, show_B in [
        (axes[0], (Ex_lab,Ey_lab), f'Lab frame S  (v=0)', False),
        (axes[1], (Ex_b,  Ey_b  ), f"Boosted frame S'  (β={beta:.2f}, γ={gamma:.2f})", True)
    ]:
        ax.set_facecolor('#16213e')
        ax.set_xlim(-3,3);ax.set_ylim(-3,3);ax.set_aspect('equal')
        ax.set_xlabel('x');ax.set_ylabel('y')
        ax.set_title(title,color=BLUE,fontsize=11)
        speed = np.sqrt(Ex**2+Ey**2)+1e-9
        ax.streamplot(x[0],y[:,0],Ex,Ey,color=RED,
                      linewidth=0.9,density=1.2,arrowsize=0.8)
        if show_B and np.any(np.abs(Bz_b)>0.01):
            # B field as color contour
            cf=ax.contourf(x,y,np.clip(Bz_b,-2,2),levels=20,
                           cmap='RdBu',alpha=0.35)
            ax.text(-2.8,-2.8,'Color: Bz field',color=DIM,fontsize=8)
        ax.plot(0,0,'o',color=YELLOW,ms=10,zorder=5)
        leg = 'E field' + ('+B(color)' if show_B else '')
        ax.text(2.5,-2.8,f'E: streamlines\nB: color',
                color=DIM,fontsize=8,ha='right')
        ax.grid(True,alpha=0.3)

    plt.suptitle(f'Lorentz boost: {config}  β={beta:.2f}  γ={gamma:.2f}',
                 color=BLUE,fontsize=12)
    plt.tight_layout(); plt.show()

interact(plot_boost,
    beta=FloatSlider(value=0.5, min=0.0, max=0.99, step=0.01, description='β = v/c'),
    config=Dropdown(options=['Point charge (E only)','Line charge (E only)'],
                    description='Setup'));

## Ch 6: Dipole Radiation Pattern & Larmor Power
$dP/d\Omega \propto \sin^2\theta$ and $P = \mu_0\omega^4 p_0^2/(12\pi c)$

In [ ]:
def plot_radiation(log_omega=1.0, p0=1.0, show_3d=False):
    omega = 10**log_omega
    fig, axes = plt.subplots(1,2,figsize=(12,5),facecolor='#1a1a2e')

    # Left: radiation pattern (polar)
    ax = axes[0]
    ax.remove()
    ax = fig.add_subplot(1,2,1,projection='polar')
    ax.set_facecolor('#16213e')
    theta = np.linspace(0,2*np.pi,500)
    r_pat = np.sin(theta)**2
    ax.plot(theta,r_pat,color=GREEN,lw=2.5)
    ax.fill(theta,r_pat,color=GREEN,alpha=0.2)
    ax.plot([np.pi/2,np.pi/2],[0,1],'--',color=DIM,lw=1)
    ax.plot([3*np.pi/2,3*np.pi/2],[0,1],'--',color=DIM,lw=1)
    ax.text(np.pi/2+0.1,1.15,'max',color=GREEN,fontsize=9)
    ax.text(0.05,0.05,'zero (along dipole axis)',
            color=RED,fontsize=8,transform=ax.transAxes)
    ax.set_rticks([0.5,1.0])
    ax.tick_params(colors=DIM,labelsize=8)
    ax.set_title(r'$dP/d\Omega \propto \sin^2\theta$',color=BLUE,pad=15,fontsize=12)
    ax.spines['polar'].set_color('#4a5a7e')

    # Right: P vs omega
    ax2 = axes[1]; ax2.set_facecolor('#16213e')
    omega_arr = np.logspace(-1,3,300)
    P_arr = omega_arr**4 * p0**2  # proportional
    ax2.loglog(omega_arr, P_arr, color=GREEN, lw=2, label=r'$P\propto\omega^4$')
    ax2.axvline(omega, color=YELLOW, lw=1.5, ls='--')
    P_now = omega**4 * p0**2
    ax2.plot(omega, P_now, 'o', color=YELLOW, ms=10, zorder=5)
    ax2.text(omega*1.2, P_now*1.5,
             f'ω={omega:.1f}\nP∝{P_now:.1e}',
             color=YELLOW,fontsize=9)
    # Compare: Rayleigh ω⁴
    ax2.fill_between(omega_arr,P_arr*0.01,P_arr,alpha=0.08,color=GREEN)
    ax2.set_xlabel('ω (rad/s)',fontsize=10)
    ax2.set_ylabel('Radiated power P (arb.)',fontsize=10)
    ax2.set_title('Larmor power: P ∝ ω⁴',color=BLUE,fontsize=12)
    ax2.legend(facecolor='#1a1a2e',labelcolor='white')
    ax2.grid(True,alpha=0.4,which='both')
    ax2.set_xlim(0.1,1000); ax2.set_ylim(1e-4,1e12)

    plt.suptitle(f'Dipole radiation  p₀={p0:.1f}  ω={omega:.1f}',
                 color=BLUE,fontsize=12)
    plt.tight_layout(); plt.show()

interact(plot_radiation,
    log_omega=FloatSlider(value=1.0, min=-1, max=3, step=0.1, description='log₁₀(ω)'),
    p0=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='p₀'));

## Ch 7: Lorentz Oscillator — Dielectric Function ε(ω)
$\epsilon(\omega) = 1 + \dfrac{f\omega_p^2}{\omega_0^2 - \omega^2 - i\gamma\omega}$

Real part ε₁: dispersion. Imaginary part ε₂: absorption. Adjust sliders to explore resonances.

In [ ]:
def plot_epsilon(omega0=1.0, gamma=0.1, f=1.0, omega_p=1.0):
    fig, axes = plt.subplots(1,3,figsize=(14,5),facecolor='#1a1a2e')

    omega = np.linspace(0.01,3.0,1000)
    denom = (omega0**2-omega**2)**2 + (gamma*omega)**2
    eps1 = 1 + f*omega_p**2*(omega0**2-omega**2)/denom
    eps2 = f*omega_p**2*gamma*omega/denom
    n_re = np.real(np.sqrt(eps1 + 1j*eps2))  # n = Re sqrt(eps)

    for ax in axes: ax.set_facecolor('#16213e')

    # ε₁ and ε₂
    ax = axes[0]
    ax.plot(omega,eps1,color=BLUE, lw=2,label='ε₁ (dispersion)')
    ax.plot(omega,eps2,color=RED,  lw=2,label='ε₂ (absorption)')
    ax.axvline(omega0,color=DIM,lw=1,ls='--',label=f'ω₀={omega0:.2f}')
    ax.axhline(0,color='#4a5a7e',lw=0.8)
    ax.axhline(1,color='#4a5a7e',lw=0.8,ls=':')
    ax.set_xlabel('ω/ω₀');ax.set_ylabel('ε(ω)')
    ax.set_title('Dielectric function',color=BLUE)
    ax.legend(facecolor='#1a1a2e',labelcolor='white',fontsize=8)
    ax.set_ylim(-3,5);ax.grid(True,alpha=0.4)
    # Mark anomalous dispersion region
    mask=(eps1<np.roll(eps1,1))&(omega>omega0*0.8)&(omega<omega0*1.5)
    ax.fill_between(omega,eps1,where=mask,alpha=0.15,color=RED,label='Anomalous')

    # Refractive index
    ax2 = axes[1]
    ax2.plot(omega,n_re,color=GREEN,lw=2)
    ax2.axvline(omega0,color=DIM,lw=1,ls='--')
    ax2.set_xlabel('ω');ax2.set_ylabel('n(ω) = Re√ε')
    ax2.set_title('Refractive index',color=BLUE)
    ax2.set_ylim(0,3);ax2.grid(True,alpha=0.4)
    ax2.text(0.05,0.92,'Normal: dn/dω>0',transform=ax2.transAxes,
             color=GREEN,fontsize=9)
    ax2.text(0.05,0.82,'Anomalous: dn/dω<0',transform=ax2.transAxes,
             color=RED,fontsize=9)

    # Absorption length
    # Intensity decay length 1/(2 k_2) with k_2 = (omega/c) Im sqrt(eps); units c = omega_0 = 1
    k2 = omega*np.imag(np.sqrt(eps1 + 1j*eps2))
    abs_plot = np.clip(1/(2*k2 + 1e-30), 1e-3, 1e5)
    ax3 = axes[2]
    ax3.semilogy(omega,abs_plot,color=YELLOW,lw=2)
    ax3.axvline(omega0,color=DIM,lw=1,ls='--')
    ax3.set_xlabel('ω');ax3.set_ylabel('Absorption length (units c/ω₀)')
    ax3.set_title('Absorption length',color=BLUE)
    ax3.set_ylim(1e-2,1e5);ax3.grid(True,alpha=0.4,which='both')
    ax3.text(omega0+0.05,3e-2,'Strong\nabsorption',color=RED,fontsize=9)

    plt.suptitle(f'Lorentz oscillator:  ω₀={omega0:.2f}  γ={gamma:.3f}  f={f:.1f}  ωₚ={omega_p:.2f}',
                 color=BLUE,fontsize=12)
    plt.tight_layout(); plt.show()

interact(plot_epsilon,
    omega0=FloatSlider(value=1.0, min=0.3, max=2.5, step=0.05, description='ω₀ (resonance)'),
    gamma= FloatSlider(value=0.1, min=0.01, max=0.8, step=0.01, description='γ (damping)'),
    f=     FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='f (strength)'),
    omega_p=FloatSlider(value=1.0, min=0.3, max=2.0, step=0.1, description='ωₚ (plasma)'));

## Summary: What Each Visualization Shows

| Visualization | Key Physics | Try this! |
|---|---|---|
| **Coulomb field** | Superposition principle, field lines ⊥ equipotentials | Set q₁=q₂>0 → repulsion; q₁=-q₂ → dipole |
| **Current loop B** | Dipole field at large r, on-axis peak | Increase R → peak broadens; compare exact vs dipole |
| **EM wave** | Transverse wave, E⊥B⊥k̂, polarization states | Sweep t/T from 0→1 to animate; try circular polarization |
| **Lorentz boost** | E and B mix under boosts, magnetism = relativistic effect | β→0.99: extreme field compression; β=0: pure Coulomb |
| **Dipole radiation** | P∝ω⁴ (Rayleigh), donut pattern | High ω → huge power; explains why blue sky scatters more |
| **ε(ω) Lorentz** | Dispersion, absorption, K-K relations | γ→0: sharp resonance; multiple resonances → complex spectrum |